# Assignment 2 — Architecture lead comparison

This notebook checks the shared SLM and prepares Part 4 using Persons 2 and 3’s actual evidence. It does not invent questions, facts or model responses.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'a7/benchmark_contract.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [2]:
import json
import torch
from a7.slm.handoff import load_selected, generate, state_hash
torch.set_num_threads(2)
manifest = json.loads((ROOT/'a7/slm/results/selected_checkpoint.json').read_text())
model, vocab = load_selected()
before = state_hash(model)
assert before == manifest['state_sha256']
sample = generate(model, vocab, 'students ', **manifest['generation'])
assert state_hash(model) == before
print('Original tensor state verified:', before)
print('Sample continuation (not an evaluation question):', repr(sample[len('students '):]))
print('Context length:', model.context_length, 'characters')
reproduction=ROOT/'a7/results/architecture/slm_reproduction_check.json'
if reproduction.exists():
    check=json.loads(reproduction.read_text(encoding='utf-8'))
    print('Independent exported-prompt matches:',check['matching_responses'],'/',check['total_responses'])


Original tensor state verified: a8b4dcabc94e13e50afc0123b4198b8394a572bc425e8898858e35cbe571e700
Sample continuation (not an evaluation question): 'artificial in partificial intelligencers context tokearmers.\nthe uning models previos.\nstartificial '
Context length: 32 characters
Independent exported-prompt matches: 25 / 25


## Part 4: baseline, RAG and fine-tuning

The table preserves actual exported continuations. Scores are assistant-assessed, not independent human grading. Baseline and RAG evidence are integrated; Person 3’s fine-tuning results are not supplied.

RAG Top-K = 1, selected by highest mean factual accuracy with ties resolved by lower K. Selection uses these same five evaluation questions; this is descriptive, not held-out evaluation.

| Question | Baseline SLM | SLM + RAG | Fine-tuned SLM |
|---|---|---|---|
| Q1: when is laundry day. | the predicts probabmenters are use models and un learning res artifining paplications.<br>students crea | smalll learning res models preveows.<br>ingrametion learaning helps useful models models can learnung o | Pending fine-tuning evidence |
| Q2: when is trash day. | the predicts probabilition paransmelld django application artificial intelligencers usefuse models l | the predicts probabilition paransmelld django application artificial intelligencers usefuse models l | Pending fine-tuning evidence |
| Q3: where are guest towels. | artificial inngual networks are pytok<br>stentraining hang helps artificial intelligencers artuage mode | artition use preversitity lon learnedicts artual ingaal networks learnedicts artificial inngerameter | Pending fine-tuning evidence |
| Q4: where are spare bulbs. | attention rkeach.<br>trach.<br>traning he students useful softworks learne predicts artuage models can re  | students unenchartificial intelligencers context tokearmers.<br>the uning models previos.<br>startificial  | Pending fine-tuning evidence |
| Q5: where is dish soap. | students expe models chartificial intelligencers con prevers artificial intelligencers.<br>attion usegr | the charaning parammechantion rkware python runing rametersing res.<br>trammechation rammechane canis.<br> | Pending fine-tuning evidence |

| Condition | Mean factual accuracy /10 | Mean relevance /10 | Responses with unsupported information /5 |
|---|---|---|---|
| Baseline | 1.00 | 1.00 | 5 |
| RAG | 1.00 | 1.00 | 5 |
| Fine-tuned | Pending | Pending | Pending |

### Retrieval and context evidence

| Top-K | Mean factual accuracy /10 | Questions retaining every expected fact in the initial visible context /5 |
|---|---|---|
| 1 | 1.00 | 2 |
| 2 | 1.00 | 0 |
| 3 | 1.00 | 0 |
| 5 | 1.00 | 0 |

4/5 selected RAG continuations differ from baseline. A changed continuation does not establish factual improvement. Original checkpoint-state hashes match; only the final 32 prompt characters reach the model at its first prediction. Retain the exported prompts/order rather than rebuilding tied retrieval rankings with different NumPy/scikit-learn versions.

### Quality and consistency evidence

- Q1, Baseline: Assistant assessment of the supplied raw continuation: Unrelated fragments about learning and models; no laundry-day answer. Expected fact: monday. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.
- Q1, RAG: Assistant assessment of the supplied raw continuation: Unrelated fragments about learning and models; no laundry-day answer. Expected fact: monday. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.
- Q2, Baseline: Assistant assessment of the supplied raw continuation: Unrelated fragments about probability, Django and models; no trash-day answer. Expected fact: friday. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.
- Q2, RAG: Assistant assessment of the supplied raw continuation: Unrelated fragments about probability, Django and models; no trash-day answer. Expected fact: friday. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.
- Q3, Baseline: Assistant assessment of the supplied raw continuation: Unrelated fragments about neural networks and training; no towel location. Expected fact: in the hall. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.
- Q3, RAG: Assistant assessment of the supplied raw continuation: Unrelated fragments about neural networks and training; no towel location. Expected fact: in the hall. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.
- Q4, Baseline: Assistant assessment of the supplied raw continuation: Unrelated fragments about attention, students and models; no bulb location. Expected fact: in the shed. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.
- Q4, RAG: Assistant assessment of the supplied raw continuation: Unrelated fragments about attention, students and models; no bulb location. Expected fact: in the shed. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.
- Q5, Baseline: Assistant assessment of the supplied raw continuation: Unrelated fragments about students, software or learning; no soap location. Expected fact: under the sink. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.
- Q5, RAG: Assistant assessment of the supplied raw continuation: Unrelated fragments about students, software or learning; no soap location. Expected fact: under the sink. Off-topic assertions/fragments are unsupported by this household knowledge base; this flag does not assert that every fragment is factually false.

Consistency here means the observed pattern across five questions. Single-seed runs do not measure stability across random seeds.

### Step 4.2 final analysis

1. Baseline, RAG achieved the highest available mean factual accuracy (1.00/10) on these five questions. A three-condition accuracy winner cannot be named until fine-tuning outputs arrive.
2. Fine-tuning is the approach that updates weights; baseline and RAG retained the original selected checkpoint state. No fine-tuned weights or outputs were supplied to verify that condition.
3. RAG is easier to maintain as facts change because knowledge documents can be updated without retraining model weights.
4. For a startup with frequently changing policies, retrieval is the maintenance-oriented choice, provided the model can use the retrieved context. Interpret this classroom SLM’s actual quality scores in light of its 32-character window.
5. RAG and fine-tuning can be combined: tune the model’s behavior and retrieve current facts at inference time. That combined condition was not tested here.

## Comparison method and pending analysis

Use the same five shared questions and generation settings: CPU, two threads, seed 42 per question, temperature 0.7, 100 new characters. Preserve exact prompts and show continuation-only responses. Score accuracy against document facts and relevance on a disclosed assistant-assessed 1–10 scale; record unsupported claims and consistency observations. Baseline/RAG weights must match the original hash; fine-tuned weights must differ. RAG context is the final 32 prompt characters.

Baseline and RAG results are supplied by Person 2; final three-condition accuracy conclusions await fine-tuning evidence from Person 3. Fine-tuning changes weights; RAG updates external knowledge. Retrieval supports changing policies without retraining, provided the model can use the retrieved context. RAG and fine-tuning can be combined, although that condition is not tested here. Single-seed runs cannot establish stability across random seeds.